# 🔥 09 — PyTorch 基础实战

> 🎯 **目标**：从 NumPy 思维丝滑过渡到 PyTorch 思维，掌握 Tensor 操作、Autograd、nn.Module、训练循环。
> ⏱️ 预计时间：1 天
> 🍎 默认环境：Apple Silicon + MPS

---

## 📋 为什么 NumPy 之后还要学 PyTorch？

| 需求 | NumPy | PyTorch |
|------|-------|---------|
| 矩阵运算 | ✅ | ✅ |
| GPU 加速 | ❌（需 CuPy） | ✅ 原生 MPS/CUDA |
| 自动求导 | ❌ 手动算梯度 | ✅ `backward()` |
| 定义神经网络 | ❌ | ✅ `nn.Module` |
| 后续 Phase 2 手写 Transformer | ❌ | ✅ 唯一选择 |

> 💡 NumPy 是你理解张量运算的"母语"，PyTorch 只是换了一个支持 GPU + 自动求导的"方言"。

## 1️⃣ Tensor 创建：和 NumPy 对应关系

> 核心原则：**你能用 NumPy 做的，几乎都能用 PyTorch 做。语法相似度 95%。**

In [ ]:
import torch
import numpy as np

# --- 创建 Tensor（对照 NumPy）---
# NumPy: np.array([1, 2, 3])
t = torch.tensor([1, 2, 3])
print(f"torch.tensor: {t}")

# NumPy: np.zeros((2, 3))
print(f"zeros:\n{torch.zeros(2, 3)}")

# NumPy: np.ones((2, 3))
print(f"ones:\n{torch.ones(2, 3)}")

# NumPy: np.random.randn(2, 3)
torch.manual_seed(42)  # 🔑 PyTorch 需要手动设随机种子
print(f"randn:\n{torch.randn(2, 3)}")

# NumPy: np.arange(0, 10, 0.5)
print(f"arange: {torch.arange(0, 10, 0.5)}")

In [ ]:
# --- 维度与形状操作 ---
x = torch.randn(2, 3, 4)  # 3 维张量
print(f"shape: {x.shape}")        # torch.Size([2, 3, 4])
print(f"dtype: {x.dtype}")        # torch.float32
print(f"ndim: {x.ndim}")          # 3
print(f"numel: {x.numel()}")      # 2×3×4 = 24（元素总数）

# 变形：.view() ≈ NumPy .reshape()
print(f"view:\n{x.view(2, 12).shape}")   # torch.Size([2, 12])
print(f"view:\n{x.view(-1, 4).shape}")   # torch.Size([6, 4])，-1 = 自动推导

# 索引切片：和 NumPy 一模一样
print(f"x[0, :, :].shape: {x[0, :, :].shape}")    # torch.Size([3, 4])
print(f"x[:, 0, 0]: {x[:, 0, 0]}")                 # 取所有 batch 的第 0 行第 0 列

In [ ]:
# --- 数学运算 ---
a = torch.tensor([[1., 2.], [3., 4.]])
b = torch.tensor([[5., 6.], [7., 8.]])

# 逐元素运算
print(f"a + b:\n{a + b}")
print(f"a * b:\n{a * b}")   # 逐元素乘（不是矩阵乘法！）

# 矩阵乘法（两种写法）
print(f"a @ b:\n{a @ b}")                        # 推荐 🔥
print(f"torch.matmul(a, b):\n{torch.matmul(a, b)}")

# 统计
print(f"sum: {x.sum()}")
print(f"mean dim=0: {x.mean(dim=0).shape}")  # 沿 batch 维求均值 → [3, 4]
print(f"argmax dim=-1:\n{x[0].argmax(dim=-1)}")  # 每行最大值的索引

## 2️⃣ Tensor 数据类型与设备（CPU / MPS / CUDA）

> 🍎 Apple Silicon 用户最关心的问题：怎么让 PyTorch 用上 MPS（Metal Performance Shaders）加速？

In [ ]:
# --- 数据类型 ---
# PyTorch 默认 float32，但大模型常用 float16/bfloat16
x32 = torch.tensor([1.0], dtype=torch.float32)   # 32 位
x16 = torch.tensor([1.0], dtype=torch.float16)   # 16 位（省显存）
xbf16 = torch.tensor([1.0], dtype=torch.bfloat16) # BF16（训练用）

print(f"float32 内存: {x32.element_size()} bytes")
print(f"float16 内存: {x16.element_size()} bytes")

# 类型转换
x = torch.randn(3, 3)
print(f"原始: {x.dtype}")
print(f"to float16: {x.half().dtype}")    # .half() = float16
print(f"to float32: {x.float().dtype}")   # .float() = float32

In [ ]:
# --- 设备管理（🍎 MPS / 🖥️ CUDA / 💻 CPU）---
# 检测可用设备
def get_device():
    if torch.cuda.is_available():
        return "cuda"
    elif torch.backends.mps.is_available():
        return "mps"  # Apple Silicon GPU
    else:
        return "cpu"

device = get_device()
print(f"✅ 使用设备: {device}")

# 创建张量时指定设备
x_cpu = torch.randn(3, 3)                       # 默认 CPU
x_mps = torch.randn(3, 3, device=device)          # 指定 MPS

# 移动张量到设备
x = torch.randn(1000, 1000)
x = x.to(device)  # 移到 MPS/CUDA

# ⚠️ MPS 注意事项
print("""
MPS 注意事项:
  1. 不是所有操作都支持 MPS（会 fallback 到 CPU）
  2. MPS 张量不能直接转 NumPy（需要先 .to("cpu")）
  3. int64 类型在 MPS 上有坑，建议模型用 float32
""")

## 3️⃣ 自动微分 Autograd——PyTorch 的灵魂

> NumPy 做不了的"自动求梯度"，是 PyTorch 最核心的能力。

In [ ]:
# --- requires_grad：标记需要跟踪的变量 ---
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2 + 3 * x + 1  # y = x² + 3x + 1

print(f"x: {x}")
print(f"y: {y}")  # tensor([11., 19.])
print(f"y.grad_fn: {y.grad_fn}")  # y 记录了自己的计算来源

# backward：自动计算 dy/dx
# dy/dx = 2x + 3 → x=2 → 7, x=3 → 9
y.sum().backward()  # 🪄 自动求梯度！
print(f"dy/dx = {x.grad}")  # tensor([7., 9.]) ✅

In [ ]:
# --- 计算图机制 ---
# 每次 backward() 梯度会累积（不清零）
x = torch.tensor([2.0], requires_grad=True)

for i in range(3):
    y = x ** 2
    y.backward()
    print(f"第{i+1}次 backward 后: x.grad = {x.grad}")
    # 第1次: 4.0
    # 第2次: 8.0（梯度累加！）
    # 第3次: 12.0

# ✅ 正确做法：手动清零或使用 optimizer.zero_grad()

# --- 不需要梯度时：torch.no_grad() ---
# 推理/评估时用，省显存、加速
with torch.no_grad():
    y = x ** 2  # 不跟踪，不建计算图
print(f"推理模式下 requires_grad: {y.requires_grad}")  # False

## 4️⃣ nn.Module——定义神经网络的标准方式

> 所有 PyTorch 模型的父类。Transformer、LoRA、GPT——最终都继承 `nn.Module`。

In [ ]:
import torch.nn as nn

# --- 最小神经网络示例 ---
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()  # 🔑 必须调用父类 __init__
        self.fc1 = nn.Linear(input_dim, hidden_dim)  # 全连接层 1
        self.fc2 = nn.Linear(hidden_dim, output_dim)  # 全连接层 2
        self.act = nn.ReLU()  # 激活函数
        
    def forward(self, x):
        """定义前向传播。PyTorch 会自动处理反向传播。"""
        x = self.fc1(x)       # [B, input] → [B, hidden]
        x = self.act(x)
        x = self.fc2(x)       # [B, hidden] → [B, output]
        return x

# 实例化
model = SimpleMLP(input_dim=10, hidden_dim=64, output_dim=2)
print(model)

# 查看参数
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数: {total_params:,}")  # 10×64 + 64 + 64×2 + 2 = 834

In [ ]:
# --- 常用层速查 ---
# nn.Linear — 全连接层（MLP/Transformer 的 FFN 都靠它）
linear = nn.Linear(512, 256)     # in_features=512, out_features=256
x = torch.randn(4, 512)          # [batch=4, features=512]
print(f"Linear: {linear(x).shape}")  # [4, 256]

# nn.Embedding — 词嵌入层（Transformer 第一层）
embed = nn.Embedding(num_embeddings=10000, embedding_dim=512)
token_ids = torch.randint(0, 10000, (4, 32))  # [batch=4, seq=32]
print(f"Embedding: {embed(token_ids).shape}")  # [4, 32, 512]

# nn.LayerNorm — 层归一化（Transformer 每层都有）
ln = nn.LayerNorm(512)
print(f"LayerNorm: {ln(x).shape}")  # [4, 512]
# 归一化后均值≈0, 标准差≈1
normalized = ln(x)
print(f"归一化后均值: {normalized.mean().item():.4f}")
print(f"归一化后标准差: {normalized.std().item():.4f}")

## 5️⃣ 损失函数与优化器

In [ ]:
# --- 损失函数 ---
# 分类问题 → CrossEntropyLoss
logits = torch.randn(4, 10)  # [batch=4, num_classes=10]
labels = torch.randint(0, 10, (4,))  # 真实标签
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)
print(f"CrossEntropyLoss: {loss.item():.4f}")

# 回归问题 → MSELoss
pred = torch.randn(4, 1)
target = torch.randn(4, 1)
mse = nn.MSELoss()(pred, target)
print(f"MSELoss: {mse.item():.4f}")

# --- 优化器 ---
model = SimpleMLP(10, 64, 2)

# SGD — 随机梯度下降
sgd = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

# Adam — 自适应学习率（最常用 🔥）
adam = torch.optim.Adam(model.parameters(), lr=0.001)

# AdamW — 解耦权重衰减（Transformer 训练首选 🔥🔥🔥）
adamw = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.01)

print(f"SGD: lr={sgd.param_groups[0]['lr']}")
print(f"Adam: lr={adam.param_groups[0]['lr']}")
print(f"AdamW: lr={adamw.param_groups[0]['lr']}, weight_decay={adamw.param_groups[0]['weight_decay']}")

## 6️⃣ 训练循环模板（最重要的一节）

> 记忆口诀：**zero_grad → forward → loss → backward → step**

In [ ]:
def train_one_epoch(model, dataloader, loss_fn, optimizer, device):
    """训练一个 epoch 的标准模板"""
    model.train()  # 🔑 训练模式（启用 dropout/batch_norm）
    total_loss = 0
    
    for batch_idx, (x, y) in enumerate(dataloader):
        x, y = x.to(device), y.to(device)  # 数据移到 GPU
        
        # 1️⃣ 清零梯度（PyTorch 默认累加，必须手动清零）
        optimizer.zero_grad()
        
        # 2️⃣ 前向传播
        output = model(x)
        loss = loss_fn(output, y)
        
        # 3️⃣ 反向传播
        loss.backward()
        
        # 4️⃣ 更新参数
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)


@torch.no_grad()  # 评估时不需要梯度
def evaluate(model, dataloader, loss_fn, device):
    """评估函数"""
    model.eval()  # 🔑 评估模式（禁用 dropout/batch_norm）
    total_loss = 0
    correct = 0
    total = 0
    
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        output = model(x)
        loss = loss_fn(output, y)
        
        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    
    return total_loss / len(dataloader), correct / total

print("✅ 训练循环模板定义完成")
print("记忆口诀：zero_grad → forward → loss → backward → step")

## 7️⃣ DataLoader 基础

In [ ]:
from torch.utils.data import Dataset, DataLoader, TensorDataset

# --- 方式 1：TensorDataset（数据已经是 Tensor，最简单）---
x = torch.randn(100, 10)
y = torch.randint(0, 2, (100,))
dataset = TensorDataset(x, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

for batch_x, batch_y in loader:
    print(f"batch_x: {batch_x.shape}, batch_y: {batch_y.shape}")
    break
# 输出: batch_x: torch.Size([16, 10]), batch_y: torch.Size([16])

# --- 方式 2：自定义 Dataset（数据来自文件/数据库）---
class MyDataset(Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

# DataLoader 核心参数
loader = DataLoader(
    dataset,
    batch_size=32,      # 批量大小
    shuffle=True,       # 每 epoch 打乱（训练集 True，验证集 False）
    num_workers=0,      # 🍎 Mac 上设为 0（多进程有坑）
    drop_last=True,     # 丢弃不完整的最后 batch
)
print(f"Batches per epoch: {len(loader)}")

## 8️⃣ NumPy ↔ PyTorch 互转

In [ ]:
# NumPy → PyTorch
np_arr = np.array([[1., 2.], [3., 4.]])
torch_tensor = torch.from_numpy(np_arr)  # 共享内存！
print(f"from_numpy: {torch_tensor}")

# PyTorch → NumPy
torch_tensor = torch.randn(3, 3)
np_arr = torch_tensor.numpy()  # ⚠️ 需要 CPU + requires_grad=False
print(f"to numpy: {type(np_arr)}")

# ⚠️ 常见陷阱
x = torch.randn(3, 3, requires_grad=True)
# x.numpy()  # ❌ RuntimeError: Can't call numpy() on Tensor that requires grad
x_detached = x.detach().numpy()  # ✅ .detach() 切断计算图

# MPS tensor 不能直接转 NumPy
device = "mps" if torch.backends.mps.is_available() else "cpu"
if device == "mps":
    x_mps = torch.randn(3, 3, device="mps")
    # x_mps.numpy()  # ❌ TypeError
    x_cpu = x_mps.cpu().numpy()  # ✅ 先移回 CPU
    print(f"MPS → CPU → NumPy: {type(x_cpu)}")
else:
    print("当前设备不是 MPS，跳过此测试")

## 9️⃣ 动手实验 1：用 PyTorch 复现线性回归

In [ ]:
torch.manual_seed(42)

# 生成数据：y = 3x₁ - 2x₂ + 1 + noise
n_samples = 200
X = torch.randn(n_samples, 2)
true_w = torch.tensor([3.0, -2.0])
true_b = 1.0
y = X @ true_w + true_b + torch.randn(n_samples) * 0.5

# 模型：最简单的 Linear Regression
model = nn.Linear(2, 1)  # 输入 2 维 → 输出 1 维
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

# 训练
losses = []
for epoch in range(100):
    pred = model(X).squeeze()
    loss = loss_fn(pred, y)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())

print(f"学到的权重: {model.weight.data}")
print(f"真实权重: {true_w}")
print(f"学到的偏置: {model.bias.data.item():.4f}")
print(f"真实偏置: {true_b}")
print(f"最终 loss: {losses[-1]:.4f}")

## 🔟 动手实验 2：MLP 二分类（非线性决策边界）

In [ ]:
from sklearn.datasets import make_moons

# 生成半月形数据（非线性可分）
X, y = make_moons(n_samples=500, noise=0.2, random_state=42)
X = torch.FloatTensor(X)
y = torch.LongTensor(y)

# 模型
class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 2),  # 二分类
        )
    
    def forward(self, x):
        return self.net(x)

model = MLPClassifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

# 训练
for epoch in range(200):
    optimizer.zero_grad()
    output = model(X)
    loss = loss_fn(output, y)
    loss.backward()
    optimizer.step()

# 评估
with torch.no_grad():
    pred = model(X).argmax(dim=1)
    acc = (pred == y).float().mean()
print(f"MLP 分类准确率: {acc:.2%}")

In [ ]:
# --- 可视化分类结果 ---
import matplotlib.pyplot as plt

with torch.no_grad():
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = torch.meshgrid(
        torch.linspace(x_min, x_max, 100),
        torch.linspace(y_min, y_max, 100),
        indexing='ij'
    )
    grid = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    Z = model(grid).argmax(dim=1).reshape(100, 100)

    plt.figure(figsize=(8, 6))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='RdYlBu', edgecolor='k', s=20)
    plt.title(f'MLP 分类决策边界（准确率: {acc:.2%}）')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.show()

## 🍎 Apple Silicon MPS 最佳实践

In [ ]:
import torch

def check_mps():
    """检查 MPS 是否可用"""
    if torch.backends.mps.is_available():
        print("✅ MPS 可用")
        x = torch.ones(3, 3, device="mps")
        print(f"   测试张量设备: {x.device}")
    else:
        print("❌ MPS 不可用")
        print("   请确认: macOS 12.3+ / PyTorch 1.12+")

check_mps()

print("""
🍎 MPS 开发注意事项:
1. int64 在 MPS 上支持不完整 → 模型/标签用 int32 或 long
2. 某些操作会 fallback 到 CPU → 可能变慢（而非报错）
3. MPS 显存 = 统一内存 → 不需要 .cuda() 式的显存管理
4. DataLoader num_workers=0 → Mac 上多进程有坑
5. torch.cuda.* 函数在 MPS 上不可用
""")

## 📝 自测题

1. `torch.tensor([1,2,3])` 和 `np.array([1,2,3])` 有什么区别？
2. `requires_grad=True` 的作用是什么？为什么推理时要用 `torch.no_grad()`？
3. 写出训练循环的四步口诀（英文）。
4. MPS tensor 能直接 `.numpy()` 吗？不能的话怎么转？
5. `model.train()` 和 `model.eval()` 的区别是什么？
6. `nn.Linear(512, 256)` 有多少个可训练参数？
7. optimizer 的 `zero_grad()` 为什么必须放在 `forward` 之前？
8. DataLoader 的 `shuffle=True` 在训练集和验证集上应该怎么设？
9. `x = x.to(device)` 做了什么？为什么每个 batch 都要调？
10. 如果你在 Mac M5 上跑 PyTorch，device 应该设为什么？

<details>
<summary>点击查看答案</summary>

1. torch.tensor 可以在 GPU/MPS 上运行，支持自动求导；np.array 只能 CPU。
2. requires_grad=True 让 PyTorch 追踪该张量的所有操作，以便 backward() 自动求梯度。推理时不需要梯度，torch.no_grad() 省显存+加速。
3. zero_grad → forward → loss → backward → step
4. 不能。需要先 `.cpu()` 移到 CPU，再 `.numpy()`。
5. train() 启用 Dropout 和 BatchNorm 的训练行为；eval() 禁用它们。
6. 512×256 + 256(bias) = 131,328 个参数。
7. PyTorch 默认累加梯度。如果不先清零，梯度会越来越大（等于之前所有 batch 的梯度和）。
8. 训练集 shuffle=True，验证集 shuffle=False。
9. 把张量从 CPU 移到 GPU/MPS。每个 batch 从 DataLoader 出来时默认在 CPU 上。
10. `device = "mps" if torch.backends.mps.is_available() else "cpu"`
</details>

---

## ✅ 产出物 Checklist

- [ ] 创建一个 Tensor，在 CPU 和 MPS 之间来回移动
- [ ] 用 autograd 计算 y = x³ + 2x 在 x=3 处的导数（答案：29）
- [ ] 定义一个自己的 nn.Module 子类（至少 2 层 Linear + 激活函数）
- [ ] 跑通"6️⃣ 训练循环模板"（复制代码运行，理解每一步）
- [ ] 用 DataLoader 加载数据并迭代一个 epoch
- [ ] 完成两个动手实验：线性回归 + MLP 分类
- [ ] 回答 10 道自测题（至少对 8 道）

---

> 🎉 完成这个 notebook 后，你就具备了 Phase 2 手写 Transformer/Attention 所需的全部 PyTorch 基础！